In [1]:
from rag_helper import RAGBase
from openai import OpenAI

In [2]:
import os

In [4]:
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

### Asking without tools

In [5]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="llama-3.3-70b-versatile",
    input=messages,
)

response.output_text

"You'd like to join a course, but I need a bit more information. Could you please provide more details about the course you're interested in, such as its name, topic, or platform? That way, I can better assist you in determining if it's still possible to join."

### Defining the tool

In [6]:
from ingest import load_faq_data, build_index

In [8]:
documents = load_faq_data()
print(f"Loaded {len(documents)} documents")
index = build_index(documents)

Loaded 1346 documents


In [9]:
def search(query):
   boost_dict = {"question": 2.0, "section": 0.5}
   filter_dict = {"course": "llm-zoomcamp"}

   return index.search(
     query, 
     boost_dict=boost_dict,
     filter_dict=filter_dict,
     num_results=5
  )

In [10]:
search_results = search("How do I run Ollama?")
search_results

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\n

In [11]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

### Sending the question with the tool

In [12]:
import json

# 1) Check messages serializability
try:
    json_str = json.dumps(messages)
    print("messages OK (len bytes):", len(json_str))
except Exception as e:
    print("messages not serializable:", type(e).__name__, e)

# 2) Check tools/search_tool serializability
try:
    json_str = json.dumps([search_tool])
    print("tools OK (len bytes):", len(json_str))
except Exception as e:
    print("tools not serializable:", type(e).__name__, e)

# 3) Inspect contents briefly
print("messages preview:", messages[:2])
print("search_tool preview keys:", list(search_tool.keys()))

messages OK (len bytes): 77
tools OK (len bytes): 319
messages preview: [{'role': 'user', 'content': 'I just discovered the course. Can I join it?'}]
search_tool preview keys: ['type', 'name', 'description', 'parameters']


In [13]:
response = openai_client.responses.create(
    model="openai/gpt-oss-120b",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseReasoningItem(id='resp_01kvdt3dcae3197q04m37qg2t9', summary=[], type='reasoning', content=[Content(text='User asks: "I just discovered the course. Can I join it?" We need to answer based on the FAQ. Likely about enrollment eligibility. We need to search FAQ for enrollment.', type='reasoning_text')], encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"join course eligibility"}', call_id='fc_d9effa6d-cbfe-4223-a2df-a906c58cb53e', name='search', type='function_call', id='fc_d9effa6d-cbfe-4223-a2df-a906c58cb53e', namespace=None, status='completed')]

### Executing the function and sending the result back

In [14]:
import json
call = response.output[1]
call
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)


In [15]:
import json
# Append the assistant's text output as a proper assistant message
messages.append({
    "role": "assistant",
    "content": response.output_text
})
# Append the tool result as a tool message so the model can consume it
messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 {'role': 'assistant', 'content': ''},
 {'type': 'function_call_output',
  'call_id': 'fc_d9effa6d-cbfe-4223-a2df-a906c58cb53e',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "bd31146b0e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "When will the course be offered next?",\n    "answer": "Summer 2027."\n  },\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workflow?",\n    "answer": "Start with the [LLM Zoomcamp d

In [16]:
import json
try:
    json.dumps(messages)
    print("messages OK")
except TypeError as e:
    print("NOT serializable:", e)

messages OK


In [18]:
response = openai_client.responses.create(
    model="llama-3.3-70b-versatile",
    input=messages,
    tools=[search_tool],
)

response.output_text

''

In [19]:
response.output

[ResponseReasoningItem(id='resp_01kvdt3zyyeadtqxj8xeaczy04', summary=[], type='reasoning', content=None, encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"join course"}', call_id='2mpahqnx2', name='search', type='function_call', id='2mpahqnx2', namespace=None, status='completed')]

### The Agentic Loop

#### A developer prompt

In [20]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

### A function-call helper

In [37]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

### Processing one response

In [29]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="llama-3.3-70b-versatile",
    input=messages,
    tools=[search_tool],
)
messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"joining the course"}


### The full agent loop

In [30]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="llama-3.3-70b-versatile",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        print("item:", type)
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
item: <class 'type'>
item: <class 'type'>
ASSISTANT:
Yes, you can still join the course. If you want to receive a certificate, you need to submit your project while they’re still accepting submissions.


### Wrapping it in a function

In [31]:
def agent_loop(instructions, question, model="llama-3.3-70b-versatile") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [32]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Running Olama locally"}
iteration #2...
ASSISTANT:
To run Olama locally, you can follow these general steps:

1. Ensure you have Python, `uv`, Jupyter, Docker, and any other necessary tools installed on your system.
2. Set up your environment to ensure reproducibility, including pinning dependency versions and documenting your Python version.
3. Create a `.env` file with your own credentials and API keys, and make sure to keep it in `.gitignore` to avoid sharing sensitive information.
4. Use a cheap model, such as `gpt-4o-mini` or Groq, to avoid burning through credits when running your project.
5. Follow the instructions in your project's README to recreate the system from scratch using your own credentials.

Note that the specific steps may vary depending on your project's requirements and setup. It's essential to document your setup and keep your environment reproducible to ensure that others can run your project successfully.


"To run Olama locally, you can follow these general steps:\n\n1. Ensure you have Python, `uv`, Jupyter, Docker, and any other necessary tools installed on your system.\n2. Set up your environment to ensure reproducibility, including pinning dependency versions and documenting your Python version.\n3. Create a `.env` file with your own credentials and API keys, and make sure to keep it in `.gitignore` to avoid sharing sensitive information.\n4. Use a cheap model, such as `gpt-4o-mini` or Groq, to avoid burning through credits when running your project.\n5. Follow the instructions in your project's README to recreate the system from scratch using your own credentials.\n\nNote that the specific steps may vary depending on your project's requirements and setup. It's essential to document your setup and keep your environment reproducible to ensure that others can run your project successfully."

In [38]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"joining the course after discovery"}
iteration #2...
ASSISTANT:
Yes, you can still join the course, but if you want to receive a certificate, you need to submit your project while they’re still accepting submissions.


'Yes, you can still join the course, but if you want to receive a certificate, you need to submit your project while they’re still accepting submissions.'

### Encouraging multiple searches

In [40]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [41]:
agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"joining the course"}
iteration #2...
ASSISTANT:
Yes, you can still join the course. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.


'Yes, you can still join the course. If you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

### Restricting off-topic questions

In [43]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"Queen Gambit"}
iteration #2...
ASSISTANT:
The Queen's Gambit is a popular chess opening that starts with the moves 1.d4 d5 2.c4. It is considered to be one of the oldest and most aggressive openings in chess, and it has been a favorite among players for centuries.

The Queen's Gambit is characterized by the move 2.c4, which attacks the Black pawn on d5 and puts pressure on the center of the board. Black can respond in several ways, but the most common response is 2...dxc4, which accepts the gambit and allows White to develop their pieces quickly.

The Queen's Gambit is known for its complex and dynamic nature, with many possible variations and sub-variations. It is a popular choice among chess players of all levels, from beginners to grandmasters, and it has been the subject of much analysis and debate over the years.

In addition to its significance in chess, the term "Queen's Gambit" has also been used as a metaphor for a strategic or t

'The Queen\'s Gambit is a popular chess opening that starts with the moves 1.d4 d5 2.c4. It is considered to be one of the oldest and most aggressive openings in chess, and it has been a favorite among players for centuries.\n\nThe Queen\'s Gambit is characterized by the move 2.c4, which attacks the Black pawn on d5 and puts pressure on the center of the board. Black can respond in several ways, but the most common response is 2...dxc4, which accepts the gambit and allows White to develop their pieces quickly.\n\nThe Queen\'s Gambit is known for its complex and dynamic nature, with many possible variations and sub-variations. It is a popular choice among chess players of all levels, from beginners to grandmasters, and it has been the subject of much analysis and debate over the years.\n\nIn addition to its significance in chess, the term "Queen\'s Gambit" has also been used as a metaphor for a strategic or tactical maneuver in other areas of life, such as business or politics. It impli

In [44]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [45]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"Queen Gambit"}
iteration #2...
ASSISTANT:
The Queen's Gambit is a popular chess opening that starts with the moves 1.d4 d5 2.c4. It is considered one of the oldest and most aggressive openings in chess, and it has been a favorite among many top players throughout history.

In the Queen's Gambit, White offers a pawn to Black in order to put pressure on the d5 pawn and gain control of the center of the board. Black can accept the gambit by playing dxc4, or decline it by playing a different move.

The Queen's Gambit is known for its complex and dynamic nature, with many possible variations and sub-variations. It is a popular choice among players of all levels, from beginners to grandmasters, and it has been the subject of much study and analysis over the years.

In addition to being a chess opening, "The Queen's Gambit" is also the title of a novel by Walter Tevis, which was published in 1983. The book tells the story of a young orphan girl 

'The Queen\'s Gambit is a popular chess opening that starts with the moves 1.d4 d5 2.c4. It is considered one of the oldest and most aggressive openings in chess, and it has been a favorite among many top players throughout history.\n\nIn the Queen\'s Gambit, White offers a pawn to Black in order to put pressure on the d5 pawn and gain control of the center of the board. Black can accept the gambit by playing dxc4, or decline it by playing a different move.\n\nThe Queen\'s Gambit is known for its complex and dynamic nature, with many possible variations and sub-variations. It is a popular choice among players of all levels, from beginners to grandmasters, and it has been the subject of much study and analysis over the years.\n\nIn addition to being a chess opening, "The Queen\'s Gambit" is also the title of a novel by Walter Tevis, which was published in 1983. The book tells the story of a young orphan girl named Beth Harmon, who becomes a chess prodigy and challenges the male-dominate